In [ ]:
# Re-installing unsloth and dependencies after runtime restart.
!pip install "unsloth[cu121] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl transformers accelerate peft bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-krburzt1/unsloth_198061ae82364e12ac73c3b4ac738c0f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-krburzt1/unsloth_198061ae82364e12ac73c3b4ac738c0f
  Resolved https://github.com/unslothai/unsloth.git to commit b96a04c17bc6bcb5522eafb17adc2b104be38f99
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 141.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 28.5 MB/s eta 0:00:

KeyboardInterrupt: 

In [ ]:
# import torch
# torch.__version__

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import pandas as pd
from datasets import Dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(device)

cuda


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [ ]:
# Set torch dtype based on CUDA capability
if torch.cuda.get_device_capability()[0] >= 8:
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

base_model = "unsloth/Llama-3.2-1B-bnb-4bit"

# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    

In [ ]:
file_path_test = '/content/drive/My Drive/001.PhD/Smartphone/traintest/testDataset2.csv'

In [ ]:
dfTest = pd.read_csv(file_path_test)

In [ ]:
dfTest.head()

,sequence
0,"android.net.wifi.RSSI_CHANGED, android.intent...."
1,"android.intent.action.DREAMING_STARTED, androi..."
2,"android.intent.action.DREAMING_STOPPED, androi..."
3,"android.net.wifi.RSSI_CHANGED, android.intent...."
4,"android.net.wifi.RSSI_CHANGED, android.intent...."


In [ ]:
sequence_list_test = dfTest['sequence'].tolist()

In [ ]:
sequence_list_test = sequence_list_test[:8000]

In [ ]:
len(sequence_list_test)

8000

In [ ]:
testDataset = []
for sequence in sequence_list_test:
    result = sequence.split(",")
    result = [item.strip() for item in result]
    testDataset.append(result)

In [ ]:
testDataset2 = []
for sequence in testDataset:
    input = " ".join(sequence[:-1])
    row = [input, sequence[-1]]
    testDataset2.append(row)

In [ ]:
# Step 1: Split into two lists
inputsTest = [pair[0] for pair in testDataset2]
targetsTest = [pair[1] for pair in testDataset2]

In [ ]:
data_dict_test = {
    "input": inputsTest,
    "target": targetsTest
}

In [ ]:
datasetTest = Dataset.from_dict(data_dict_test)

In [ ]:
instruction = """You are a Smartphone Event Predictor agent named John.
Analyze the input and predict the target.
"""

tokenizer.chat_template = "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{{'<|begin_of_text|>' + message['role'] + '\n' + message['content'] + '<|end_of_text|>\n'}}{% endfor %}{% if add_generation_prompt %}{{'<|begin_of_text|>assistant\n'}}{% endif %}"


In [ ]:
predictedTagetsTest = []

for i in range(8000):
    messages = [{"role": "system", "content": instruction},
      {"role": "user", "content": datasetTest['input'][i]}]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to("cuda")

    outputs = model.generate(**inputs, max_new_tokens=150, num_return_sequences=1)

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    predictedEvent = text.split("assistant")[1].lstrip("\n")

    predictedTagetsTest.append(predictedEvent)

In [ ]:
# print(predictedTagetsTest[7999])

In [ ]:
# print(predictedTagetsTest[1])

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
precision = precision_score(targetsTest, predictedTagetsTest, average='micro')
recall = recall_score(targetsTest, predictedTagetsTest, average='micro')
f1_weighted = f1_score(targetsTest, predictedTagetsTest, average='weighted')
f1_macro = f1_score(targetsTest, predictedTagetsTest, average='macro')
print("Precision:", precision)
print("Recall:", recall)
print("F1-weighted:", f1_weighted)
print("F1-macro:", f1_macro)

Precision: 0.0
Recall: 0.0
F1-weighted: 0.0
F1-macro: 0.0
